# Exercise 06: Vector Search

> **Chapter:** Ch06 - Vector Search
> **Estimated time:** ~40 minutes
>
> This exercise is optional. No submission, no grading.
> The solution notebook is released one week after the exercise.

A real vector query rarely asks only for the nearest neighbors. It asks for the nearest neighbors
that also satisfy a condition: movies after 2005, documents of one customer, products in stock.
The chapter describes three ways to combine a filter with an approximate index and argues about
their costs. In this exercise you measure them. You implement post-filtering, filtering during the
search, and pre-filtering on an inverted file over 100,000 word vectors. You then run them against
filters of very different selectivity and build a search that picks the right strategy for each
query. Along the way you meet a filter that breaks the chapter's cost estimate, and you work out why.

**Your tasks** (tick them off as you go):

- [ ] **Quiz** - 20 questions in the quiz app
- [ ] 💻 **C1: Three ways to filter** - implement post-, during-, and pre-filtering in `tasks.py`
- [ ] ✏ **T1: A filter that follows the content** - explain why one filter defeats the inverted file
- [ ] 💻 **C2: Choosing per query** - a search that picks its strategy and meets a recall and cost budget

---

## Quiz

Open the quiz app and work through the **20 questions** for this chapter. On the start screen,
pick the topic **"06 - Vector Search"**:

**Quiz app:** https://roger-weber.github.io/mmir-unibasel-hs26/quiz/

The quiz covers definitions and basic concepts. The tasks below go further: you measure what the
chapter only estimates, and you design a filtering rule that holds up on filters the estimate
gets wrong.

---

## Word vectors with metadata

You work with the GloVe word vectors of the chapter demos: 100 dimensions, trained on Wikipedia
and news text. We keep the 100,000 most frequent words and normalize every vector to unit length,
so the dot product equals the cosine similarity. As queries, we hold out 200 words drawn from the
20,000 most frequent ones. The index holds the remaining 99,800 words.

> **First run downloads data.** The GloVe file is about 130 MB. If you ran demo ch05-02 or ch06-02,
> it is already cached under `~/gensim-data/` and loads in a few seconds.

In [ ]:
# Standard setup - run this first
import sys, pathlib, re

# Make `shared/` importable regardless of notebook depth
# (exercise notebooks live in exercises/chNN/, solutions in exercises/chNN/solution/).
_here = pathlib.Path().resolve()
_p = _here
while not (_p / "shared").is_dir() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))       # the shared/ root, for `from shared...`
sys.path.insert(0, str(_here))    # the notebook's own folder wins for `import tasks`

%load_ext autoreload
%autoreload 2

import numpy as np

from shared.data import load_glove_vectors
from shared.display import display_md, print_table

words, vectors = load_glove_vectors()
words, vectors = words[:100_000], vectors[:100_000]
vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

rng = np.random.default_rng(0)
query_rows = rng.choice(20_000, size=200, replace=False)
index_rows = np.setdiff1d(np.arange(len(words)), query_rows)
Q, X = vectors[query_rows], vectors[index_rows]
query_words = [words[i] for i in query_rows]
index_words = [words[i] for i in index_rows]
display_md(f"**Index:** {len(X):,} words x {X.shape[1]} dimensions. **Queries:** {len(Q)} held-out words.")

Every word now receives two pieces of metadata, so that we can filter on them.

- **A tenant.** Imagine a service that hosts the vectors of 10,000 customers in one shared index.
  Every vector belongs to exactly one customer, its tenant, and a query may only see the vectors of
  certain tenants. We assign the tenants at random, so they say nothing about a word's meaning.
  Filtering on 8%, 1%, or 0.1% of the tenants, or on a single one, gives filters of known
  selectivity.
- **A content attribute.** A word either contains a digit or it does not: `1990`, `6-4`, `b52`,
  `2nd`. This is a property of the word itself, like the genre of a movie or the language of a
  document.

Each filter is a boolean array over the index, `True` for the admitted words. This is the bitmap
of the chapter.

In [ ]:
tenant = np.random.default_rng(1).integers(0, 10_000, size=len(X))
has_digit = np.array([bool(re.search(r"\d", w)) for w in index_words])

FILTERS = {
    "8% of tenants":    tenant < 800,
    "1% of tenants":    tenant < 100,
    "0.1% of tenants":  tenant < 10,
    "one tenant":       tenant == 0,
    "contains a digit": has_digit,
}
print_table([[name, f"{mask.sum():,}", f"{mask.mean():.2%}"] for name, mask in FILTERS.items()],
            headers=["Filter", "Admitted words", "Share of the index"])

## The index you will search

Your code lives in <a href="tasks.py">tasks.py</a>, a small `FilteredIVF` class. The inverted file
itself is **already built for you**. The constructor runs k-means with 256 centers and stores one
list of vector ids per center, as in the chapter. Four helpers are provided:

- `probe(q, n_probe)` returns the ids of the `n_probe` lists whose centers are most similar to `q`.
- `score(q, ids)` returns the dot products of `q` with the vectors `ids`.
- `top_k(ids, sims, k)` returns the `k` ids with the highest similarity, best first.
- `search(q, k, n_probe)` is the plain IVF search without a filter.

The class also counts its work. Every similarity it computes, against a center or a vector, adds 1
to `index.scored`. Checking a bit of the filter is free. This is the cost model of the chapter: the
filter bitmap is cheap, and the distance computations are what we want to save. Four methods are
left as stubs for you: `search_post`, `search_during`, `search_pre`, and `search_auto`.

Build the index once. K-means over 99,800 vectors takes a few seconds.

In [ ]:
from tasks import FilteredIVF

index = FilteredIVF(X)
sizes = [len(l) for l in index.lists]
display_md(f"**Index built:** {index.n_list} lists, between {min(sizes):,} and {max(sizes):,} "
           f"vectors per list (median {int(np.median(sizes)):,}).")

To judge a filtered search, we need the right answer: the exact top 10 among the admitted vectors.
The helpers below compute it independently of your code. They also define recall for filters that
admit fewer than 10 vectors, and they measure a strategy over all 200 queries. We use 16 probed
lists throughout, which is 6% of the 256 lists.

In [ ]:
K, N_PROBE = 10, 16
FULL_SCAN = len(X)

def exact_filtered(q, admitted, k=K):
    """Exact top k among the admitted vectors (the reference answer)."""
    ids = np.flatnonzero(admitted)
    return ids[np.argsort(-(X[ids] @ q), kind="stable")[:k]]

TRUTH = {name: [exact_filtered(q, mask) for q in Q] for name, mask in FILTERS.items()}

def recall(found, truth):
    """Share of the exact filtered top k that was found (1.0 if nothing is admitted)."""
    return len(set(found) & set(truth)) / len(truth) if len(truth) else float(len(found) == 0)

def measure(strategy, name):
    """Mean recall@10, mean number of results, and mean cost of `strategy` on one filter.

    strategy(q, admitted) must return an array of ids; its cost is read from index.scored.
    """
    rec, returned, cost = [], [], []
    for q, truth in zip(Q, TRUTH[name]):
        index.scored = 0
        found = strategy(q, FILTERS[name])
        cost.append(index.scored)
        rec.append(recall(found, truth))
        returned.append(len(found))
    return float(np.mean(rec)), float(np.mean(returned)), float(np.mean(cost))

# Baseline: the plain IVF search without a filter, against the exact unfiltered top 10.
FILTERS["no filter"] = np.ones(len(X), dtype=bool)
TRUTH["no filter"] = [exact_filtered(q, FILTERS["no filter"]) for q in Q]
r, _, c = measure(lambda q, a: index.search(q, K, N_PROBE), "no filter")
del FILTERS["no filter"], TRUTH["no filter"]
display_md(f"**Plain IVF, 16 of 256 lists, no filter:** recall@10 = {r:.2f} at {c:,.0f} "
           f"similarities per query, {c / FULL_SCAN:.1%} of a full scan ({FULL_SCAN:,}).")

## Three ways to filter

> **How this works:** Implement `search_post`, `search_during`, and `search_pre` in
> <a href="tasks.py">tasks.py</a>. Each docstring is the spec. Write the code yourself or direct an
> AI from the docstrings, then run the verify cell. It either passes or fails with a specific
> message. Autoreload picks up each save, so you never restart the kernel.

### 💻 Task C1 - Post-filtering, filtering during the search, and pre-filtering

The chapter names three strategies. **Post-filtering** runs the vector search first and then
removes the results that fail the filter. **Filtering during the search** scans the probed lists
as usual but skips every vector the filter rejects, before computing its similarity.
**Pre-filtering** ignores the index and compares the query exactly with every admitted vector.

Post-filtering comes in two variants. If the filter runs inside the index, it can see every
candidate the scan has scored. Often it runs in the application instead. The application calls
the index, which computes all these similarities internally but returns only its top `fetch`
results, and the application filters those. FAISS's `index.search(q, fetch)` and an HNSW search
with its candidate list of length $ef_{\text{search}}$ both behave like this. Your `search_post`
covers both cases through its `fetch` parameter.

Implement all three strategies so that the verify cell passes, then run the explore cell and read
the table.

In [ ]:
# --- Verify C1 ---
q = Q[0]
for name, mask in FILTERS.items():
    for label, found in [("post", index.search_post(q, K, mask, N_PROBE)),
                         ("during", index.search_during(q, K, mask, N_PROBE)),
                         ("pre", index.search_pre(q, K, mask))]:
        assert found is not None, f"{label}: the method returns None, did you forget `return`?"
        assert len(found) <= K, f"{label} on '{name}': at most k results, got {len(found)}"
        assert mask[found].all(), f"{label} on '{name}': returned a vector the filter rejects"

# Pre-filtering is exact, and it scores exactly the admitted vectors.
for name, mask in FILTERS.items():
    index.scored = 0
    found = index.search_pre(q, K, mask)
    assert list(found) == list(TRUTH[name][0]), f"pre on '{name}': not the exact filtered top 10"
    assert index.scored == mask.sum(), \
        f"pre on '{name}': scored {index.scored}, expected one per admitted vector ({mask.sum()})"

# Filtering during the search scores the centers plus the admitted entries of the probed lists.
mask = FILTERS["1% of tenants"]
probed = np.argsort(-(index.centers @ q), kind="stable")[:N_PROBE]
entries = np.concatenate([index.lists[j] for j in probed])
index.scored = 0
found = index.search_during(q, K, mask, N_PROBE)
expected = index.n_list + mask[entries].sum()
assert index.scored == expected, \
    f"during: scored {index.scored}, expected {expected} (256 centers + admitted entries only)"
assert set(found) <= set(entries), "during: every result must come from the probed lists"

# Post-filtering filters the plain search, so its cost does not depend on the filter.
index.scored = 0
plain = index.search(q, 100, N_PROBE)
plain_cost = index.scored
index.scored = 0
found = index.search_post(q, K, mask, N_PROBE, fetch=100)
assert index.scored == plain_cost, f"post: scored {index.scored}, expected {plain_cost}"
assert list(found) == [i for i in plain if mask[i]][:K], \
    "post: keep the admitted candidates of the plain search, in their order"
assert list(index.search_post(q, K, np.ones(len(X), bool), N_PROBE)) == list(index.search(q, K, N_PROBE)), \
    "post: without a filter it must equal the plain search"

# With fetch=None, post-filtering sees all scored candidates of the probed lists.
for name, mask in FILTERS.items():
    index.scored = 0
    found = index.search_post(q, K, mask, N_PROBE)
    assert index.scored == index.n_list + len(entries), \
        f"post (fetch=None) on '{name}': scored {index.scored}, expected the full plain search"
    assert len(found) == min(K, mask[entries].sum()), \
        f"post (fetch=None) on '{name}': filter all {len(entries):,} scanned candidates, not only the top k"

# Edge cases: a filter that admits nothing, and k larger than the admitted set.
empty = np.zeros(len(X), dtype=bool)
for label, found in [("post", index.search_post(q, K, empty, N_PROBE)),
                     ("during", index.search_during(q, K, empty, N_PROBE)),
                     ("pre", index.search_pre(q, K, empty))]:
    assert len(found) == 0, f"{label}: an empty filter must return no results"
one = FILTERS["one tenant"]
assert len(index.search_pre(q, 50, one)) == one.sum(), \
    f"pre: with k = 50 and {one.sum()} admitted vectors, return all of them"

print("✓ all three filtering strategies behave correctly!")

In [ ]:
# --- Explore C1 ---
STRATEGIES = {
    "post (all scanned)": lambda q, a: index.search_post(q, K, a, N_PROBE),
    "post (fetch 100)": lambda q, a: index.search_post(q, K, a, N_PROBE, fetch=100),
    "post (fetch 10)":  lambda q, a: index.search_post(q, K, a, N_PROBE, fetch=K),
    "during":           lambda q, a: index.search_during(q, K, a, N_PROBE),
    "pre":              lambda q, a: index.search_pre(q, K, a),
}
rows = []
for name, mask in FILTERS.items():
    for label, strategy in STRATEGIES.items():
        r, n, c = measure(strategy, name)
        rows.append([name, f"{mask.mean():.2%}", label, f"{r:.2f}", f"{n:.1f}", f"{c:,.0f}"])
print_table(rows, headers=["Filter", "Admitted", "Strategy", "Recall@10", "Results", "Similarities"])
# For each filter: which strategy would you pick, and what does it cost compared with a full scan?
# Compare "post (all scanned)" with "during": what differs, and what does not?
# Compare "1% of tenants" with "contains a digit" in the "during" rows.

## A filter that follows the content

The cells below look inside the index. They use only the provided parts of the class, so they run
even before you finish C1. The first one compares two tenant groups with the digit filter. For
each filter it reports three numbers:

- the share of admitted words in the whole index,
- the median share of admitted words in the 16 lists that a query probes,
- the median position, in the probe order, of the lists that hold the exact filtered top 10. A
  position up to 16 means the search scans that list.

In [ ]:
list_of = np.empty(len(X), dtype=int)
for j, members in enumerate(index.lists):
    list_of[members] = j

def probe_order(q):
    return np.argsort(-(index.centers @ q), kind="stable")

rows = []
for name in ["8% of tenants", "1% of tenants", "contains a digit"]:
    mask, local, positions = FILTERS[name], [], []
    for q, truth in zip(Q, TRUTH[name]):
        order = probe_order(q)
        position = np.empty(index.n_list, dtype=int)
        position[order] = np.arange(1, index.n_list + 1)
        local.append(mask[np.concatenate([index.lists[j] for j in order[:N_PROBE]])].mean())
        positions.extend(position[list_of[truth]])
    rows.append([name, f"{mask.mean():.1%}", f"{np.median(local):.1%}",
                 f"{np.median(positions):.0f}", f"{np.mean(np.array(positions) > N_PROBE):.0%}"])
print_table(rows, headers=["Filter", "Share in the index", "Median share in probed lists",
                           "Median probe position of the true neighbors", "True neighbors beyond 16"])

The second cell shows how the digit words are spread over the 256 lists, and three example
queries. For each query, it lists the exact digit neighbors and the digit words that the 16 probed
lists offer instead.

In [ ]:
mask = FILTERS["contains a digit"]
share = np.array([mask[members].mean() for members in index.lists])
in_digit_lists = mask[np.concatenate([index.lists[j] for j in np.flatnonzero(share > 0.5)])].sum()
display_md(f"**Digit words per list:** median share {np.median(share):.1%}. "
           f"{(share > 0.5).sum()} of {index.n_list} lists are more than half digit words, "
           f"and they hold {in_digit_lists:,} of the {mask.sum():,} digit words.")

def digit_words_in_probed(q):
    ids = np.concatenate([index.lists[j] for j in probe_order(q)[:N_PROBE]])
    ids = ids[mask[ids]]
    return ids[np.argsort(-(X[ids] @ q), kind="stable")]

offered = [digit_words_in_probed(q) for q in Q]
counts = np.array([len(o) for o in offered])
found = np.array([len(set(t) & set(o[:K])) / K for t, o in zip(TRUTH["contains a digit"], offered)])
examples = [int(np.argmin(counts)),                                   # fewest digit words offered
            int(np.argmin(np.where(counts >= 100, found, np.inf))),   # plenty offered, worst recall
            int(np.argmax(counts))]                                   # most digit words offered
rows = []
for i in examples:
    truth = TRUTH["contains a digit"][i]
    rows.append([query_words[i], f"{counts[i]:,}",
                 ", ".join(index_words[j] for j in truth[:4]),
                 ", ".join(index_words[j] for j in offered[i][:4]),
                 f"{found[i]:.1f}"])
print_table(rows, headers=["Query", "Digit words in probed lists", "Exact digit neighbors",
                           "Best digit words in probed lists", "Recall@10"])

> **How this works:** Write your answer in the markdown cell below the question (replace
> *Your answer here...*). The solution notebook fills the same cell with a model answer, so you can
> compare and reflect.

### ✏ Task T1 - Why the digit filter defeats the inverted file

The digit filter admits about 8% of the index, the same share as the 8% tenant group. The chapter
estimates the admitted candidates of a filtered IVF search as the admitted share times the entries
of the probed lists. By that estimate, the two filters should behave alike. In your C1 table,
filtering during the search reaches a recall of about 0.9 for the tenant group and about 0.6 for
the digit filter, less than for the 1% tenant group.

Answer three questions. First, explain what goes wrong for the digit filter. Base your explanation
on how the inverted file assigns vectors to lists, and use the numbers and example queries above.
Second, explain why the chapter's estimate works for the tenant filters but not for this one. Third,
name one filter from a real application that would behave like the digit filter, and one that would
behave like a tenant filter.

> *Your answer here...*

---

## Choosing per query

No single strategy wins on every filter. Production systems therefore decide per query, as the
chapter describes for Lucene and for systems that filter during the search. Your table from C1
contains the costs and recalls such a decision has to weigh.

> **How this works:** Implement `search_auto` in <a href="tasks.py">tasks.py</a>. It may call your
> three strategies and the provided helpers. Its docstring lists the requirements that the verify
> cell checks on every filter.

### 💻 Task C2 - A search that picks its strategy

Write `search_auto(q, k, admitted, n_probe)`. For each query, it decides how to answer, and the
result must meet the budget in the docstring: a mean recall@10 of at least 0.9 on every filter, at
less than 10% of the cost of a full scan, and below 2,500 similarities per query for the 8% tenant
group. Run the verify cell, adjust your rule until it passes, then compare your search with the
fixed strategies in the explore cell.

In [ ]:
# --- Verify C2 ---
auto = lambda q, a: index.search_auto(q, K, a, N_PROBE)
assert auto(Q[0], FILTERS["1% of tenants"]) is not None, \
    "search_auto returns None, did you forget `return`?"

rows, problems = [], []
for name, mask in FILTERS.items():
    r, n, c = measure(auto, name)
    rows.append([name, f"{r:.2f}", f"{n:.1f}", f"{c:,.0f}", f"{c / FULL_SCAN:.1%}"])
    if r < 0.9:
        problems.append(f"'{name}': recall@10 = {r:.2f}, needs at least 0.90")
    if c >= 0.1 * FULL_SCAN:
        problems.append(f"'{name}': {c:,.0f} similarities per query, needs fewer than {0.1 * FULL_SCAN:,.0f}")
    if n < min(K, mask.sum()):
        problems.append(f"'{name}': only {n:.1f} results on average, needs {min(K, mask.sum())}")
    if name == "8% of tenants" and c >= 2_500:
        problems.append(f"'{name}': {c:,.0f} similarities per query, needs fewer than 2,500")
print_table(rows, headers=["Filter", "Recall@10", "Results", "Similarities", "vs. full scan"])
for q in Q[:20]:
    for name, mask in FILTERS.items():
        assert mask[auto(q, mask)].all(), f"'{name}': search_auto returned a vector the filter rejects"
assert not problems, "search_auto misses the budget:\n  " + "\n  ".join(problems)
print("✓ search_auto meets the recall and cost budget on every filter!")

In [ ]:
# --- Explore C2 ---
# Cost of each fixed strategy that reaches recall 0.9, against your search_auto.
rows = []
for name in FILTERS:
    cells = []
    for label, strategy in {**STRATEGIES, "auto": auto}.items():
        r, n, c = measure(strategy, name)
        cells.append(f"{c:,.0f}" if r >= 0.9 else "-")
    rows.append([name] + cells)
print_table(rows, headers=["Filter"] + list(STRATEGIES) + ["auto"])
display_md("A dash means the strategy misses recall 0.9 on this filter. "
           "Which signal does your rule use to decide, and what does it cost to obtain?")